# New Section

In [ ]:
# install
!pip install ultralytics roboflow

# get the dataset fresh, inside Colab (no path mess)
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
dataset = rf.workspace("viren-dhanwani").project("tennis-ball-detection").version(6).download("yolov8")

# train -- device=0 is the GPU
!yolo detect train data={dataset.location}/data.yaml model=yolov5xu.pt epochs=100 imgsz=640 device=0

# CRITICAL: save weights to Drive before the session ends
from google.colab import drive
drive.mount('/content/drive')
!cp -r runs/detect/train/weights /content/drive/MyDrive/tennis_ball_weights

In [ ]:
# Dataset download removed: the original command contained private browser session cookies.
# Download the court keypoint dataset from its source and place it in data/.


In [ ]:
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import os
import json

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TCD(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, 'r') as f:
            self.data = json.load(f)
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = cv2.imread(os.path.join(f"{self.img_dir}/{item['id']}.png"))

        h, w = img.shape[:2]
        img = self.transform(img)
        kps = (np.array(item['kps']).flatten()).astype(np.float32)

        kps[::2] * 224 / w
        kps[1::2] * 224 / h

        return img, kps

In [ ]:
train_data = TCD('data/images', 'data/data_train.json')
val_data = TCD('data/images', 'data/data_val.json')

train_loader = DataLoader(train_data, batch_size = 8, shuffle= True)
val_loader = DataLoader(val_data, batch_size = 8, shuffle= True)

In [ ]:
model = models.resnet50(pretrained = True)
model.fc = torch.nn.Linear(model.fc.in_features, 28)
model = model.to(device)

regulation = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
epochs = 20
for epoch in range(epochs):
    for i, (img, kps) in enumerate(train_loader):
        img, kps = img.to(device), kps.to(device)
        optimizer.zero_grad()
        outputs = model(img)
        loss = regulation(outputs, kps)
        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"Epoch {epoch}, iter {i}, loss: {loss.item()}")



In [ ]:
torch.save(model.state_dict(), "tennis_court_model.pth")